# Strategy Benchmark

Extracts the top-N best performing strategies from `model_comparison_results.csv` (fine-tuned models only), then re-runs each on the FinAgent benchmark stocks and date range.

**Reusable**: re-run this notebook any time `model_comparison_results.csv` is updated — it always picks up the latest best strategies.

### Installations

In [105]:
%pip install backtrader alpaca_trade_api plotly pandas -q

Note: you may need to restart the kernel to use updated packages.


### Config

In [106]:
ALPACA_API_KEY    = ''
ALPACA_SECRET_KEY = ''

RESULTS_CSV       = 'model_comparison_results_merged.csv'

# FinAgent benchmark
BENCHMARK_SYMBOLS = ['AAPL', 'AMZN', 'MSFT', 'TSLA', 'GOOGL']
BENCHMARK_START   = '2022-06-01'
BENCHMARK_END     = '2024-01-01'

# Which fine-tuned models to extract strategies from
TARGET_MODELS = [
    'Qwen2.5-7B LoRA v1-500',
    'Llama-3.1-8B LoRA v1-500',
    'Qwen2.5-32B LoRA v1-500',
]

# How many top strategies to extract per model (ranked by return_pct)
TOP_N = 10

### Imports & Backtrader

In [107]:
import ast
import json
import re
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout

import backtrader as bt
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from alpaca_trade_api.rest import REST, TimeFrame

In [108]:
class Backtrader:
    _data_cache = {}

    def __init__(self):
        self.rest_api = REST(
            ALPACA_API_KEY, ALPACA_SECRET_KEY,
            'https://paper-api.alpaca.markets'
        )

    def _get_bars(self, symbol, start, end):
        key = (symbol, start, end)
        if key not in Backtrader._data_cache:
            Backtrader._data_cache[key] = self.rest_api.get_bars(
                symbol, TimeFrame.Day, start, end, adjustment='all'
            ).df
        return Backtrader._data_cache[key]

    def prefetch(self, symbols, start, end):
        print(f'Pre-fetching {len(symbols)} symbols ({start} to {end})...')
        for i, sym in enumerate(symbols, 1):
            self._get_bars(sym, start, end)
            print(f'  [{i}/{len(symbols)}] {sym}')
        print('Done.')

    def run_backtest(self, strategy_cls, symbol, start, end, cash=10000):
        cerebro = bt.Cerebro(stdstats=True)
        cerebro.broker.setcash(cash)
        cerebro.addstrategy(strategy_cls)
        cerebro.addanalyzer(bt.analyzers.SharpeRatio,  _name='sharpe')
        cerebro.addanalyzer(bt.analyzers.AnnualReturn, _name='annual')
        cerebro.addanalyzer(bt.analyzers.DrawDown,     _name='drawdown')
        cerebro.adddata(bt.feeds.PandasData(
            dataname=self._get_bars(symbol, start, end), name=symbol
        ))
        init_val = cerebro.broker.getvalue()
        results  = cerebro.run()
        ret      = (cerebro.broker.getvalue() / init_val - 1) * 100
        strat    = results[0]
        sharpe   = strat.analyzers.sharpe.get_analysis().get('sharperatio')
        annual   = strat.analyzers.annual.get_analysis()
        avg_ann  = (sum(annual.values()) / len(annual) * 100) if annual else 0.0
        max_dd   = strat.analyzers.drawdown.get_analysis()['max']['drawdown']
        return ret, sharpe, avg_ann, max_dd

    def timed_backtest(self, strategy_cls, symbol, start, end, timeout=10):
        with ThreadPoolExecutor(max_workers=1) as ex:
            future = ex.submit(self.run_backtest, strategy_cls, symbol, start, end)
            try:
                return future.result(timeout=timeout)
            except FuturesTimeout:
                raise TimeoutError(f'Backtest timed out ({symbol})')


bt_instance = Backtrader()
bt_instance.prefetch(BENCHMARK_SYMBOLS, BENCHMARK_START, BENCHMARK_END)

Pre-fetching 5 symbols (2022-06-01 to 2024-01-01)...
  [1/5] AAPL
  [2/5] AMZN
  [3/5] MSFT
  [4/5] TSLA
  [5/5] GOOGL
Done.


### Extract Best Strategies from CSV

In [109]:
df_results = pd.read_csv(RESULTS_CSV)

extracted = []
for model in TARGET_MODELS:
    for symbol in BENCHMARK_SYMBOLS:
        candidates = df_results[
            (df_results['model']         == model)  &
            (df_results['symbol']        == symbol) &
            (df_results['status']        == 'profitable') &
            (df_results['strategy_code'].notna())
        ].sort_values('avg_annual_return_pct', ascending=False)

        if candidates.empty:
            print(f'  SKIP {model} [{symbol}]: no profitable strategies found')
            continue

        for rank, (_, row) in enumerate(candidates.head(TOP_N).iterrows(), 1):
            extracted.append({
                'model':      model,
                'symbol':     symbol,
                'rank':       rank,
                'origin_arr': row['avg_annual_return_pct'],
                'origin_shr': row['sharpe_ratio'],
                'code':       row['strategy_code'],
                'label':      f"{model} [{symbol}] #{rank}",
            })

print(f'\nExtracted {len(extracted)} candidates (top {TOP_N} per model × symbol):\n')
for s in extracted:
    print(f"  {s['label']}  origin ARR={s['origin_arr']:.1f}%")


  SKIP Llama-3.1-8B LoRA v1-500 [AAPL]: no profitable strategies found

Extracted 100 candidates (top 10 per model × symbol):

  Qwen2.5-7B LoRA v1-500 [AAPL] #1  origin ARR=11.3%
  Qwen2.5-7B LoRA v1-500 [AAPL] #2  origin ARR=11.3%
  Qwen2.5-7B LoRA v1-500 [AAPL] #3  origin ARR=7.8%
  Qwen2.5-7B LoRA v1-500 [AAPL] #4  origin ARR=5.7%
  Qwen2.5-7B LoRA v1-500 [AAPL] #5  origin ARR=5.7%
  Qwen2.5-7B LoRA v1-500 [AAPL] #6  origin ARR=5.7%
  Qwen2.5-7B LoRA v1-500 [AAPL] #7  origin ARR=5.6%
  Qwen2.5-7B LoRA v1-500 [AAPL] #8  origin ARR=2.9%
  Qwen2.5-7B LoRA v1-500 [AMZN] #1  origin ARR=14.9%
  Qwen2.5-7B LoRA v1-500 [AMZN] #2  origin ARR=14.9%
  Qwen2.5-7B LoRA v1-500 [AMZN] #3  origin ARR=14.9%
  Qwen2.5-7B LoRA v1-500 [AMZN] #4  origin ARR=14.1%
  Qwen2.5-7B LoRA v1-500 [AMZN] #5  origin ARR=10.6%
  Qwen2.5-7B LoRA v1-500 [AMZN] #6  origin ARR=5.6%
  Qwen2.5-7B LoRA v1-500 [AMZN] #7  origin ARR=4.8%
  Qwen2.5-7B LoRA v1-500 [MSFT] #1  origin ARR=14.7%
  Qwen2.5-7B LoRA v1-500 [MSFT] #

### Run on Benchmark Symbols

In [110]:
def load_strategy(code):
    namespace = {'bt': bt}
    exec(code, namespace)
    return namespace['Strategy']


all_results = []

for strat_info in extracted:
    symbol = strat_info['symbol']
    try:
        strategy_cls = load_strategy(strat_info['code'])
    except Exception as e:
        print(f"LOAD ERROR {strat_info['label']}: {e}")
        all_results.append({
            'strategy': strat_info['label'], 'model': strat_info['model'],
            'symbol': symbol, 'rank': strat_info['rank'],
            'origin_arr': strat_info['origin_arr'],
            'return_pct': None, 'sharpe_ratio': None,
            'avg_annual': None, 'max_drawdown': None, 'status': 'error',
        })
        continue

    print(f"Running: {strat_info['label']}", end='  ')
    try:
        ret, sharpe, avg_ann, max_dd = bt_instance.timed_backtest(
            strategy_cls, symbol, BENCHMARK_START, BENCHMARK_END
        )
        status = 'profitable' if ret > 0 else ('no_trades' if ret == 0 and sharpe is None else 'loss')
        print(f"ARR={avg_ann:.1f}%  SHR={sharpe}  MDD={max_dd:.1f}%  [{status}]")
    except Exception as e:
        ret, sharpe, avg_ann, max_dd, status = None, None, None, None, 'error'
        print(f"ERROR — {str(e)[:80]}")

    all_results.append({
        'strategy':     strat_info['label'],
        'model':        strat_info['model'],
        'symbol':       symbol,
        'rank':         strat_info['rank'],
        'origin_arr':   strat_info['origin_arr'],
        'return_pct':   ret,
        'sharpe_ratio': sharpe,
        'avg_annual':   avg_ann,
        'max_drawdown': max_dd,
        'status':       status,
    })

df_all = pd.DataFrame(all_results)

# Keep best per model × symbol: profitable > loss > no_trades/error, then highest avg_annual
status_priority = {'profitable': 0, 'loss': 1, 'no_trades': 2, 'error': 3}
df_all['_sp'] = df_all['status'].map(status_priority).fillna(3)
df_bench = (
    df_all
    .sort_values(['model', 'symbol', '_sp', 'avg_annual'], ascending=[True, True, True, False])
    .groupby(['model', 'symbol'])
    .first()
    .reset_index()
    .drop(columns='_sp')
)

print(f'\nDone. {len(df_all)} candidates → {len(df_bench)} best results.')


Running: Qwen2.5-7B LoRA v1-500 [AAPL] #1  ARR=25.4%  SHR=1.340502006858562  MDD=15.6%  [profitable]
Running: Qwen2.5-7B LoRA v1-500 [AAPL] #2  ARR=25.2%  SHR=1.3443698092183263  MDD=15.6%  [profitable]
Running: Qwen2.5-7B LoRA v1-500 [AAPL] #3  ARR=0.8%  SHR=-0.008381979584730657  MDD=27.2%  [loss]
Running: Qwen2.5-7B LoRA v1-500 [AAPL] #4  ARR=3.7%  SHR=0.10465846642483038  MDD=23.6%  [profitable]
Running: Qwen2.5-7B LoRA v1-500 [AAPL] #5  ARR=3.7%  SHR=0.10465846642483038  MDD=23.6%  [profitable]
Running: Qwen2.5-7B LoRA v1-500 [AAPL] #6  ARR=3.7%  SHR=0.10465846642483038  MDD=23.6%  [profitable]
Running: Qwen2.5-7B LoRA v1-500 [AAPL] #7  ARR=3.9%  SHR=0.11182169114233079  MDD=23.6%  [profitable]
Running: Qwen2.5-7B LoRA v1-500 [AAPL] #8  ARR=10.7%  SHR=0.9065193411483164  MDD=14.8%  [profitable]
Running: Qwen2.5-7B LoRA v1-500 [AMZN] #1  ARR=14.9%  SHR=0.518056431732826  MDD=20.6%  [profitable]
Running: Qwen2.5-7B LoRA v1-500 [AMZN] #2  ARR=14.9%  SHR=0.518056431732826  MDD=20.6%  

### Results Table

In [111]:
for metric, col in [
    ('ARR — Avg Annual Return %', 'avg_annual'),
    ('SHR — Sharpe Ratio',        'sharpe_ratio'),
    ('MDD — Max Drawdown %',      'max_drawdown'),
]:
    pivot = df_bench.pivot_table(
        index='model', columns='symbol', values=col
    ).round(3)
    pivot['Avg'] = pivot.mean(axis=1).round(3)
    pivot = pivot.sort_values('Avg', ascending=(col == 'max_drawdown'))
    print(f'\n=== {metric} ===')
    display(pivot)


=== ARR — Avg Annual Return % ===


symbol,AAPL,AMZN,GOOGL,MSFT,TSLA,Avg
model,,,,,,
Qwen2.5-32B LoRA v1-500,25.227,25.096,27.406,25.989,25.413,25.826
Qwen2.5-7B LoRA v1-500,25.431,19.027,16.112,23.261,12.851,19.336
Llama-3.1-8B LoRA v1-500,NaN,0.238,0.085,4.848,1.044,1.554



=== SHR — Sharpe Ratio ===


symbol,AAPL,AMZN,GOOGL,MSFT,TSLA,Avg
model,,,,,,
Qwen2.5-7B LoRA v1-500,1.341,1.617,0.938,0.703,0.922,1.104
Qwen2.5-32B LoRA v1-500,1.344,0.488,0.964,0.788,0.346,0.786
Llama-3.1-8B LoRA v1-500,NaN,-1.721,-9.416,0.277,0.004,-2.714



=== MDD — Max Drawdown % ===


symbol,AAPL,AMZN,GOOGL,MSFT,TSLA,Avg
model,,,,,,
Llama-3.1-8B LoRA v1-500,NaN,0.627,0.176,13.727,17.046,7.894
Qwen2.5-7B LoRA v1-500,15.567,22.156,13.017,25.596,15.724,18.412
Qwen2.5-32B LoRA v1-500,15.567,41.407,12.874,26.649,63.549,32.009


### Save Results

In [112]:
df_bench.to_csv('strategy_benchmark_results.csv', index=False)
print('Saved to strategy_benchmark_results.csv')
display(df_bench[['model', 'symbol', 'avg_annual', 'sharpe_ratio', 'max_drawdown', 'status']].rename(columns={
    'avg_annual':   'ARR %',
    'sharpe_ratio': 'SHR',
    'max_drawdown': 'MDD %',
}).round(3))

Saved to strategy_benchmark_results.csv


,model,symbol,ARR %,SHR,MDD %,status
0,Llama-3.1-8B LoRA v1-500,AMZN,0.238,-1.721,0.627,profitable
1,Llama-3.1-8B LoRA v1-500,GOOGL,0.085,-9.416,0.176,profitable
2,Llama-3.1-8B LoRA v1-500,MSFT,4.848,0.277,13.727,profitable
3,Llama-3.1-8B LoRA v1-500,TSLA,1.044,0.004,17.046,profitable
4,Qwen2.5-32B LoRA v1-500,AAPL,25.227,1.344,15.567,profitable
5,Qwen2.5-32B LoRA v1-500,AMZN,25.096,0.488,41.407,profitable
6,Qwen2.5-32B LoRA v1-500,GOOGL,27.406,0.964,12.874,profitable
7,Qwen2.5-32B LoRA v1-500,MSFT,25.989,0.788,26.649,profitable
8,Qwen2.5-32B LoRA v1-500,TSLA,25.413,0.346,63.549,profitable
9,Qwen2.5-7B LoRA v1-500,AAPL,25.431,1.341,15.567,profitable
